<a href=https://github.com/yamikumo-DSD/chat_cmr/tree/main><img src="images/github-mark-white.png" width="50px"></img></a>
<font size="10px"> Knowledge DB Builder for Document Search Tool</font>

In [1]:
from lib.utils import ignore_warnings
from lib.knowledge_db import *
from global_settings import *
from agent_tools import doc_search

In [2]:
from lib.utils import ignore_warnings
from lib.knowledge_db import *
from global_settings import *
from agent_tools import doc_search
from IPython.display import HTML
import lib.uis as uis
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
import pandas as pd

In [3]:
def is_valid_path(text):
    """
    Check if the text is a valid path

    Args:
        text (str): Text to check

    Returns:
        dict: {
            'is_valid': bool,  # Whether the format is valid as a path
            'exists': bool,     # Whether it actually exists on filesystem
            'absolute': bool,   # Whether it's an absolute path
            'message': str       # Detailed message
        }
    """
    import os
    from pathlib import Path

    if not isinstance(text, str) or not text.strip():
        return {
            "is_valid": False,
            "exists": False,
            "absolute": False,
            "message": "Empty string or non-string input",
        }

    # Check path format based on OS (simplified version)
    is_valid_format = True
    message_parts = []

    if os.name == "nt":  # Windows
        if len(text) > 1 and text[1] == ":":
            # Drive letter (C:, D: etc.)
            if not text[0].isalpha():
                is_valid_format = False
                message_parts.append("Invalid drive letter")
        elif "\\" in text:
            # Contains backslash
            pass
    else:  # Unix-like systems
        if text.startswith("/"):
            # Absolute path
            pass
        elif "/" in text:
            # Relative path
            pass

    # Invalid path format
    if not is_valid_format:
        return {
            "is_valid": False,
            "exists": False,
            "absolute": False,
            "message": " ".join(message_parts) or "Invalid path format",
        }

    # Check if it's an absolute path
    is_absolute = os.path.isabs(text)

    try:
        # Check if the path exists
        path_exists = os.path.exists(text)

        if path_exists:
            message_parts.append("Exists")
        else:
            message_parts.append("Does not exist")

    except (OSError, TypeError) as e:
        path_exists = False
        message_parts.append(f"Cannot access: {str(e)}")

    return {
        "is_valid": True,
        "exists": path_exists,
        "absolute": is_absolute,
        "message": " ".join(message_parts),
    }

In [4]:
output = widgets.Output()
target_inputbox = widgets.Text(description="Target Dir")
db_name_inputbox = widgets.Text(description="DB Name", value="kdb")
build_button = widgets.Button(description="Build", layout=Layout(width="60px"), disabled=True)
save_path_inputbox = widgets.Text(description="Save Dir", value=KNOWLEDGE_DB_DIR)
chunk_size_inputbox = widgets.IntText(description="Chunk Size", value=1000, min=1)

@uis.value_change_method(target_inputbox)
@output.capture()
def target_inputbox_method(arg) -> None:
    if is_valid_path(target_inputbox.value)["exists"]:
        build_button.disabled = False
    else:
        build_button.disabled = True
        
@uis.value_change_method(save_path_inputbox)
@output.capture()
def save_path_inputbox_method(arg) -> None:
    if is_valid_path(save_path_inputbox.value)["is_valid"]:
        build_button.disabled = False
    else:
        build_button.disabled = True

@uis.button_method(build_button)
@output.capture()
def build_button_method(arg) -> None:
    build_button.disabled = True
    output.clear_output()
    try:
        build_knowledge_db(
            directory_path=target_inputbox.value,
            db_path=save_path_inputbox.value,
            name=db_name_inputbox.value,
            chunk_size=chunk_size_inputbox.value,
        )
    except BaseException as e:
        print(f"An error occured while building DB ({str(e)}).")
    build_button.disabled = False

In [5]:
test_query_inputbox = widgets.Text(description="Test Query")
query_number_chooser = widgets.IntSlider(description="N", min=1, max=10)
search_button = widgets.Button(description="Search")
test_output = widgets.Output()

@uis.button_method(search_button)
@test_output.capture()
def search_button_method(arg) -> None:
    from lib.rag import JinaRerankerMultilingual
    import pandas as pd
    
    search_button.disabled = True
    test_output.clear_output()
    
    chunks = pick_relevant_local_documents(
        query=test_query_inputbox.value,
        db_path=save_path_inputbox.value,
        name=db_name_inputbox.value,
        reranker=JinaRerankerMultilingual(),
        n_relevant_chunks=query_number_chooser.value,
        n_search_results=50,
    )
    display(pd.DataFrame(chunks))
    
    search_button.disabled = False

In [6]:
def render_builder_ui() -> None:
    display(HTML("Restart and re-run this notebook after you attempt to delete and recreate existing DB."))
    display(target_inputbox)
    display(save_path_inputbox)
    display(db_name_inputbox)
    display(HBox([chunk_size_inputbox, build_button]))
    display(output)

def render_tester_ui() -> None:
    display(test_query_inputbox)
    display(query_number_chooser)
    display(search_button)
    display(test_output)

In [7]:
render_builder_ui()

Text(value='', description='Target Dir')

Text(value='knowledge_db', description='Save Dir')

Text(value='kdb', description='DB Name')

Output()

In [8]:
render_tester_ui()

Text(value='', description='Test Query')

IntSlider(value=1, description='N', max=10, min=1)

Button(description='Search', style=ButtonStyle())

Output()